# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Notebook 6 – Hybrid Root Extraction for Unmatched Terms (DD-007)

## Objective

This notebook implements the second stage of the proposed Arabic root retrieval framework by processing the query terms that could not be matched using the Quranic Arabic Corpus (QAC).

The notebook combines rule-based linguistic normalization with the CAMeL Tools morphological analyzer to derive candidate Arabic roots for unmatched Modern Standard Arabic words and other lexical items outside the lexical coverage of the QAC.

Unlike the QAC-based retrieval stage, which relies on manually validated Qur'anic lexical annotations, this stage performs automatic morphological analysis and applies a root selection strategy to identify the most suitable candidate root for each unmatched query term.

The extracted roots are then merged with the validated QAC-derived roots to produce a more comprehensive root representation for each QuranQA question. These results serve as the input to the subsequent semantic enrichment stage using Maqāyīs al-Lughah.

In [1]:
# ============================================================
# Configure Project Path
# ============================================================

import os
import sys
from pathlib import Path


def find_workspace_root(start=None):
    """
    Locate the current thesis workspace without hard-coding a
    user-specific absolute path.
    """

    configured = os.environ.get("THESIS_WORKSPACE_ROOT")

    if configured:
        configured_path = Path(configured).expanduser().resolve()

        if (configured_path / "02_Methodology_Evidence").is_dir():
            return configured_path

        raise FileNotFoundError(
            "THESIS_WORKSPACE_ROOT is set, but it does not contain "
            "'02_Methodology_Evidence'."
        )

    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "02_Methodology_Evidence").is_dir():
            return candidate

        expected_child = (
            candidate /
            "Master Thesis - Quranic Passage Retrieval - Writing Workspace"
        )

        if (expected_child / "02_Methodology_Evidence").is_dir():
            return expected_child

    raise FileNotFoundError(
        "Could not locate the thesis workspace. Start Jupyter from the "
        "workspace (or one of its subfolders), or set THESIS_WORKSPACE_ROOT."
    )


PROJECT_ROOT = find_workspace_root()

METHODOLOGY_DIR = PROJECT_ROOT / "02_Methodology_Evidence"

QAC_ROOT_EXTRACTION_DIR = (
    METHODOLOGY_DIR /
    "QAC and Root Extraction"
)

if str(QAC_ROOT_EXTRACTION_DIR) not in sys.path:
    sys.path.insert(0, str(QAC_ROOT_EXTRACTION_DIR))

print("Project Root:")
print(PROJECT_ROOT)

print("\nQAC / Root Extraction Directory:")
print(QAC_ROOT_EXTRACTION_DIR)


Project Root:
${THESIS_WORKSPACE}

QAC / Root Extraction Directory:
${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction


In [2]:
# ============================================================
# Import Required Libraries
# ============================================================

# Standard library
import ast
import json
from pathlib import Path
from collections import Counter
from importlib.metadata import version, PackageNotFoundError

# Data manipulation
import pandas as pd
import numpy as np

# Display
from IPython.display import display

# CAMeL Tools
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

# Project module
import hybrid_root_extractor as hre

try:
    CAMEL_TOOLS_VERSION = version("camel-tools")
except PackageNotFoundError:
    CAMEL_TOOLS_VERSION = "UNKNOWN"

print(f"CAMeL Tools package version : {CAMEL_TOOLS_VERSION}")
print("CAMeL morphology database   : built-in database")


CAMeL Tools package version : 1.6.0
CAMeL morphology database   : built-in database


In [3]:

# Load the built-in morphology database
db = MorphologyDB.builtin_db()

# Create the analyzer
analyzer = Analyzer(db)

print("CAMeL Morphology Analyzer loaded successfully.")

CAMeL Morphology Analyzer loaded successfully.


## Configuring the Notebook Environment

This notebook uses consistent display settings to improve the readability of intermediate and final outputs generated during the hybrid root extraction process.

The configuration ensures that Arabic words, candidate analyses, selected roots, and evaluation results are displayed clearly throughout the notebook.

In [4]:
# ============================================================
# Configure Display Settings
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 1200)

print("=" * 70)
print("DISPLAY SETTINGS CONFIGURED")
print("=" * 70)

DISPLAY SETTINGS CONFIGURED


In [5]:
# ============================================================
# Define Project Paths
# ============================================================

CORRECTED_OUTPUT_DIR = (
    QAC_ROOT_EXTRACTION_DIR /
    "Corrected_Outputs" /
    "Point_2_Query_Root_Provenance"
)

CORRECTED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Corrected QAC output from Notebook 5
QAC_RESULTS_FILE = (
    CORRECTED_OUTPUT_DIR /
    "QuranQA_QAC_Root_Retrieval_Corrected.csv"
)

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Input File        : {QAC_RESULTS_FILE}")
print(f"Output Directory  : {CORRECTED_OUTPUT_DIR}")

print("\nValidation")
print(f"Input exists      : {QAC_RESULTS_FILE.exists()}")


PROJECT PATHS
Project Root      : ${THESIS_WORKSPACE}
Input File        : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance\QuranQA_QAC_Root_Retrieval_Corrected.csv
Output Directory  : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance

Validation
Input exists      : True


## Loading the QAC Root Retrieval Results

This notebook begins by loading the output generated by the QAC-based root retrieval stage (Notebook 5).

The dataset contains the lexical retrieval results for each QuranQA question, including the query terms, lexical matches, and validated QAC roots. These results provide the basis for identifying the query terms that remain unmatched and therefore require additional morphological analysis.

In [6]:
# ============================================================
# Load QAC Root Retrieval Results
# ============================================================

qac_results_df = pd.read_csv(QAC_RESULTS_FILE)

print("=" * 70)
print("QAC ROOT RETRIEVAL RESULTS LOADED")
print("=" * 70)

print(f"Total Questions : {len(qac_results_df):,}")
print(f"Total Columns   : {len(qac_results_df.columns)}")

print("\nColumns")
print(list(qac_results_df.columns))

# ------------------------------------------------------------
# Convert JSON strings back to Python objects
# ------------------------------------------------------------

qac_results_df["Query_Terms"] = (
    qac_results_df["Query_Terms"]
    .apply(json.loads)
)

qac_results_df["Query_Term_Provenance"] = (
    qac_results_df["Query_Term_Provenance"]
    .apply(json.loads)
)

qac_results_df["Lexical_Matches"] = (
    qac_results_df["Lexical_Matches"]
    .apply(json.loads)
)

# Ensure the corrected QAC output is being used.
required_match_fields = {
    "Original_Term",
    "Normalized_Term",
    "Lexical_Matched",
    "Root_Resolved",
    "QAC_Status",
    "QAC_Candidate_Roots"
}

sample_fields = set()

for matches in qac_results_df["Lexical_Matches"]:
    if matches:
        sample_fields = set(matches[0].keys())
        break

missing_match_fields = required_match_fields - sample_fields

if missing_match_fields:
    raise ValueError(
        "Notebook 6 requires the corrected Notebook-5 provenance fields. "
        "Missing: " + ", ".join(sorted(missing_match_fields))
    )

print("\nSample Record")
display(qac_results_df.head(1))


QAC ROOT RETRIEVAL RESULTS LOADED
Total Questions : 251
Total Columns   : 6

Columns
['Dataset', 'Question_ID', 'Question', 'Query_Terms', 'Query_Term_Provenance', 'Lexical_Matches']

Sample Record


,Dataset,Question_ID,Question,Query_Terms,Query_Term_Provenance,Lexical_Matches
0,Train,101,من هم قوم شعيب؟,"[قوم, شعيب]","[{'Token_Position': 2, 'Original_Term': 'قوم', 'Normalized_Term': 'قوم'}, {'Token_Position': 3, 'Original_Term': 'شعيب', 'Normalized_Term': 'شعيب'}]","[{'Query_Term': 'قوم', 'Original_Term': 'قوم', 'Normalized_Term': 'قوم', 'Lookup_Type': 'Word', 'Matched': True, 'Lexical_Matched': True, 'Root_Re..."


## Extracting the Terms Requiring CAMeL Fallback

This notebook processes every query term for which the QAC stage did **not** provide an acceptable root.

This includes:

- terms with no QAC lexical match,
- QAC lexical matches for which no root annotation is available, and
- normalized QAC lookup forms whose candidate entries imply conflicting roots.

This distinction is necessary because a lexical match or morphological analysis is not, by itself, evidence that a usable root has been resolved. The extracted records retain the original term, normalized query term, QAC lookup status, and QAC candidate-root evidence before CAMeL analysis is attempted.


In [7]:
# ============================================================
# Extract Query Terms Requiring CAMeL Fallback
# ============================================================

fallback_records = []

for _, row in qac_results_df.iterrows():

    for match in row["Lexical_Matches"]:

        if not match["Root_Resolved"]:

            fallback_records.append({

                "Dataset": row["Dataset"],
                "Question_ID": row["Question_ID"],
                "Question": row["Question"],

                "Query_Term": match["Query_Term"],
                "Original_Term": match["Original_Term"],
                "Normalized_Term": match["Normalized_Term"],

                "QAC_Lexical_Matched": match["Lexical_Matched"],
                "QAC_Lookup_Type": match["Lookup_Type"],
                "QAC_Status": match["QAC_Status"],
                "QAC_Candidate_Count": match["QAC_Candidate_Count"],
                "QAC_Candidate_Words": match["QAC_Candidate_Words"],
                "QAC_Candidate_Lemmas": match["QAC_Candidate_Lemmas"],
                "QAC_Candidate_Roots": match["QAC_Candidate_Roots"],
                "QAC_Candidate_POS": match["QAC_Candidate_POS"]

            })

fallback_df = pd.DataFrame(fallback_records)

print("=" * 70)
print("CAMeL FALLBACK TERMS EXTRACTED")
print("=" * 70)

print(f"Total Fallback Terms : {len(fallback_df):,}")

if len(fallback_df):

    print(
        f"Unique Terms         : "
        f"{fallback_df['Normalized_Term'].nunique():,}"
    )

    print("\nQAC Fallback Reasons")
    display(
        fallback_df["QAC_Status"]
        .value_counts()
        .rename_axis("QAC_Status")
        .reset_index(name="Count")
    )

    print("\nFirst Ten Fallback Terms")
    display(fallback_df.head(10))


CAMeL FALLBACK TERMS EXTRACTED
Total Fallback Terms : 817
Unique Terms         : 517

QAC Fallback Reasons


,QAC_Status,Count
0,NO_MATCH,743
1,LEXICAL_MATCH_NO_ROOT,50
2,AMBIGUOUS_ROOT,24



First Ten Fallback Terms


,Dataset,Question_ID,Question,Query_Term,Original_Term,Normalized_Term,QAC_Lexical_Matched,QAC_Lookup_Type,QAC_Status,QAC_Candidate_Count,QAC_Candidate_Words,QAC_Candidate_Lemmas,QAC_Candidate_Roots,QAC_Candidate_POS
0,Train,101,من هم قوم شعيب؟,شعيب,شعيب,شعيب,True,Word,LEXICAL_MATCH_NO_ROOT,2,"[شُعَيْبٌ, شُعَيْبُ]",[شُعَيْب],[],[PN]
1,Train,102,من هم قوم موسى؟,موسي,موسى,موسي,True,Word,LEXICAL_MATCH_NO_ROOT,4,"[مُوسَى, مُوسَىٰ, مُّوسَى, مُّوسَىٰ]",[مُوسَىٰ],[],[PN]
2,Train,105,من هو النبي المعروف بالصبر؟,بالصبر,بالصبر,بالصبر,False,-,NO_MATCH,0,[],[],[],[]
3,Train,106,من كفل السيدة مريم؟,السيدة,السيدة,السيدة,False,-,NO_MATCH,0,[],[],[],[]
4,Train,106,من كفل السيدة مريم؟,مريم,مريم,مريم,True,Word,LEXICAL_MATCH_NO_ROOT,2,"[مَرْيَمَ, مَرْيَمُ]",[مَرْيَم],[],[PN]
5,Train,107,ما معنى الحطمة؟,معني,معنى,معني,False,-,NO_MATCH,0,[],[],[],[]
6,Train,108,من هو اخو سيدنا موسى؟,سيدنا,سيدنا,سيدنا,False,-,NO_MATCH,0,[],[],[],[]
7,Train,108,من هو اخو سيدنا موسى؟,موسي,موسى,موسي,True,Word,LEXICAL_MATCH_NO_ROOT,4,"[مُوسَى, مُوسَىٰ, مُّوسَى, مُّوسَىٰ]",[مُوسَىٰ],[],[PN]
8,Train,109,ما معنى القارعة؟,معني,معنى,معني,False,-,NO_MATCH,0,[],[],[],[]
9,Train,110,ما معنى الجاثية؟,معني,معنى,معني,False,-,NO_MATCH,0,[],[],[],[]


## Rule-Based Morphological Normalization

Before applying automatic morphological analysis, the unmatched query terms undergo a lightweight rule-based normalization process.

The normalization reduces superficial orthographic and morphological variation while preserving the standard Arabic spelling expected by the CAMeL Tools morphological analyzer.

The normalization performs the following operations:

- Normalization of Alef variants (أ، إ، آ → ا).
- Normalization of Alef Maqsura (ى → ي).
- Removal of common attached prefix–article combinations (وال، بال، كال، فال، لل).
- Conservative removal of the definite article (ال) only when appropriate.
- Preservation of common prepositional forms such as **إليه** that are not prefixed with the definite article.

The normalization strategy was refined empirically after evaluating the retrieval performance on the QuranQA dataset to maximize compatibility with the CAMeL morphological lexicon.

In [8]:
# ============================================================
# Rule-Based Morphological Normalization
# ============================================================

# Preserve the Notebook-4 normalized query term and record the additional
# conservative normalization used specifically for CAMeL analysis.
fallback_df["CAMeL_Normalized_Term"] = (
    fallback_df["Normalized_Term"]
    .apply(hre.normalize_for_camel)
)

modified_df = fallback_df[
    fallback_df["Normalized_Term"]
    != fallback_df["CAMeL_Normalized_Term"]
].copy()

print("=" * 70)
print("RULE-BASED CAMeL NORMALIZATION COMPLETED")
print("=" * 70)

print(f"Total Fallback Terms : {len(fallback_df):,}")
print(f"Terms Modified       : {len(modified_df):,}")

if len(fallback_df):
    print(
        f"Modification Rate    : "
        f"{100 * len(modified_df) / len(fallback_df):.2f}%"
    )

print("\nSample Normalization Results")

display(
    fallback_df[
        [
            "Original_Term",
            "Normalized_Term",
            "CAMeL_Normalized_Term"
        ]
    ].head(20)
)


RULE-BASED CAMeL NORMALIZATION COMPLETED
Total Fallback Terms : 817
Terms Modified       : 304
Modification Rate    : 37.21%

Sample Normalization Results


,Original_Term,Normalized_Term,CAMeL_Normalized_Term
0,شعيب,شعيب,شعيب
1,موسى,موسي,موسي
2,بالصبر,بالصبر,صبر
3,السيدة,السيدة,سيدة
4,مريم,مريم,مريم
5,معنى,معني,معني
6,سيدنا,سيدنا,سيدنا
7,موسى,موسي,موسي
8,معنى,معني,معني
9,معنى,معني,معني


## Evaluating the Normalization Stage

The effectiveness of the rule-based normalization process is evaluated by comparing the original and normalized query terms.

This evaluation reports:

- the number of modified terms,
- the percentage of terms affected by normalization, and
- representative examples of the transformations applied.

These statistics provide evidence of the contribution of the normalization stage before morphological analysis is performed.

In [9]:
# ============================================================
# Evaluate Rule-Based Normalization
# ============================================================

modified_df = fallback_df[
    fallback_df["Query_Term"] != fallback_df["Normalized_Term"]
].copy()

print("=" * 70)
print("NORMALIZATION EVALUATION")
print("=" * 70)

print(f"Total Terms          : {len(fallback_df):,}")
print(f"Modified Terms       : {len(modified_df):,}")
print(f"Unchanged Terms      : {len(fallback_df)-len(modified_df):,}")

print(
    f"Modification Rate    : "
    f"{100*len(modified_df)/len(fallback_df):.2f}%"
)

print("\nSample Modified Terms")

display(
    modified_df[
        ["Query_Term", "Normalized_Term"]
    ].head(20)
)

NORMALIZATION EVALUATION
Total Terms          : 817
Modified Terms       : 0
Unchanged Terms      : 817
Modification Rate    : 0.00%

Sample Modified Terms


,Query_Term,Normalized_Term


## Exploring CAMeL Morphological Analyses

Before implementing the hybrid root extraction algorithm, the normalized query terms are analyzed using the CAMeL Tools morphological analyzer.

CAMeL may produce multiple analyses for a single word, each containing lexical and morphological information such as the lemma, root (when available), stem, and part-of-speech.

This exploratory step examines the analyses returned for a representative sample of unmatched query terms and provides the basis for designing the root selection strategy implemented in the subsequent stage.

In [10]:
# ============================================================
# Explore CAMeL Morphological Analyses
# ============================================================

# Load the built-in morphology database
db = MorphologyDB.builtin_db()

# Create the analyzer
analyzer = Analyzer(db)

print("=" * 70)
print("CAMeL MORPHOLOGICAL ANALYSIS")
print("=" * 70)

print(f"CAMeL Tools Version : {CAMEL_TOOLS_VERSION}")
print("Database            : built-in morphology database")

# Representative sample
sample_terms = [
    "صبر",
    "سيدة",
    "معنى",
    "سيدنا",
    "عقوبة",
    "ملكة",
    "عقروا"
]

for term in sample_terms:

    print("\n" + "=" * 60)
    print(f"Word : {term}")
    print("=" * 60)

    analyses = analyzer.analyze(term)

    print(f"Number of Analyses : {len(analyses)}")

    if len(analyses) == 0:
        print("No analysis found.")
        continue

    print("\nFirst Analysis\n")

    for key, value in analyses[0].items():
        print(f"{key:20} : {value}")


CAMeL MORPHOLOGICAL ANALYSIS
CAMeL Tools Version : 1.6.0
Database            : built-in morphology database

Word : صبر
Number of Analyses : 8

First Analysis

diac                 : صَبْر
lex                  : صَبْر
bw                   : صَبْر/NOUN
gloss                : patience;endurance
pos                  : noun
prc3                 : 0
prc2                 : 0
prc1                 : 0
prc0                 : 0
per                  : na
asp                  : na
vox                  : na
mod                  : na
stt                  : i
cas                  : u
enc0                 : 0
rat                  : i
source               : lex
form_gen             : m
form_num             : s
d3seg                : صَبْر
caphi                : s._a_b_r
d1tok                : صَبْر
d2tok                : صَبْر
pos_logprob          : -0.4344233
d3tok                : صَبْر
d2seg                : صَبْر
pos_lex_logprob      : -4.587637
num                  : s
ud                   : NOUN


## Implementing the Hybrid Root Extraction Function

Based on the exploratory analysis of the CAMeL morphological analyzer, a reusable root extraction function is implemented.

The function combines the rule-based normalization stage with CAMeL morphological analysis and applies the existing scoring criteria to the candidate analyses. The corrected implementation additionally records the selection evidence and treats a root as resolved only when a usable root is actually present.

For tied highest-scoring analyses, the root is accepted when the tied analyses agree on the same valid root. If equally preferred analyses imply conflicting roots, the term is marked ambiguous rather than being resolved by analyzer return order.

For each query term, the function preserves the selected lexical evidence, root-resolution status, root, lemma, stem, part-of-speech, candidate counts, and selection score.


In [11]:
# ============================================================
# Test the Hybrid Root Extraction Function
# ============================================================

test_words = [
    "بالصبر",
    "السيدة",
    "معنى",
    "سيدنا",
    "ملكة",
    "عقوبة",
    "عقروا"
]

results = []

for word in test_words:
    results.append(
        hre.extract_root_with_camel(word)
    )

results_df = pd.DataFrame(results)

print("=" * 70)
print("HYBRID ROOT EXTRACTION TEST")
print("=" * 70)

display(results_df)


HYBRID ROOT EXTRACTION TEST


,Query_Term,Normalized_Term,Matched,Root_Resolved,Analysis_Found,Analysis_Count,Analysis_Score,Top_Analysis_Count,Top_Roots,Top_Root_Validations,Root_Ambiguous,Resolution_Status,Root_AR,CAMeL_Root_Raw,CAMeL_Root_Normalized,CAMeL_Root_Valid,CAMeL_Root_Invalid_Reason,Lemma_AR,Stem_AR,POS,Root_Source
0,بالصبر,صبر,True,True,True,8,170,6,[صبر],"[{'Raw': 'ص.ب.ر', 'Normalized': 'صبر', 'Valid': True, 'Invalid_Reason': None}]",False,RESOLVED,صبر,ص.ب.ر,صبر,True,NaN,صَبْر,صَبْر,noun,CAMeL
1,السيدة,سيدة,True,True,True,22,165,1,[سدة],"[{'Raw': 'سّدة', 'Normalized': 'سدة', 'Valid': True, 'Invalid_Reason': None}]",False,RESOLVED,سدة,سّدة,سدة,True,NaN,سَيِّدَة,سَيِّدَة,noun_prop,CAMeL
2,معنى,معني,False,False,True,7,70,1,[],"[{'Raw': 'ع.ن.#', 'Normalized': 'عن#', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",False,ANALYSIS_INVALID_ROOT,NaN,ع.ن.#,عن#,False,incomplete_placeholder_root,مَعْنَى,مَعْنَى,noun,NaN
3,سيدنا,سيدنا,False,False,True,8,25,4,[],"[{'Raw': 'س.#.د', 'Normalized': 'س#د', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",False,ANALYSIS_INVALID_ROOT,NaN,س.#.د,س#د,False,incomplete_placeholder_root,سَيِّد,سَيِّد,noun_prop,NaN
4,ملكة,ملكة,True,True,True,28,170,14,[ملك],"[{'Raw': 'م.ل.ك', 'Normalized': 'ملك', 'Valid': True, 'Invalid_Reason': None}]",False,RESOLVED,ملك,م.ل.ك,ملك,True,NaN,مَلَكَة,مَلَك,noun,CAMeL
5,عقوبة,عقوبة,True,True,True,7,170,7,[عقب],"[{'Raw': 'ع.ق.ب', 'Normalized': 'عقب', 'Valid': True, 'Invalid_Reason': None}]",False,RESOLVED,عقب,ع.ق.ب,عقب,True,NaN,عُقُوبَة,عُقُوب,noun,CAMeL
6,عقروا,عقروا,True,True,True,2,120,2,[عقر],"[{'Raw': 'ع.ق.ر', 'Normalized': 'عقر', 'Valid': True, 'Invalid_Reason': None}]",False,RESOLVED,عقر,ع.ق.ر,عقر,True,NaN,عَقُر,عَقُر,verb,CAMeL


In [12]:
# ============================================================
# Inspect Candidate Analyses
# ============================================================

inspection_words = [
    "سيدة",
    "معنى",
    "ملكة",
    "سيدنا"
]

for word in inspection_words:

    print("\n" + "=" * 80)
    print(f"WORD : {word}")
    print("=" * 80)

    analyses = analyzer.analyze(word)

    print(f"Number of Analyses : {len(analyses)}\n")

    for i, analysis in enumerate(analyses[:10], start=1):

        print(f"Analysis {i}")

        print(
            f"Root  : {analysis.get('root')}\n"
            f"Lemma : {analysis.get('lex')}\n"
            f"Stem  : {analysis.get('stem')}\n"
            f"POS   : {analysis.get('pos')}"
        )

        print("-" * 40)


WORD : سيدة


Number of Analyses : 22

Analysis 1
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun
----------------------------------------
Analysis 2
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun
----------------------------------------
Analysis 3
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun
----------------------------------------
Analysis 4
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun
----------------------------------------
Analysis 5
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun
----------------------------------------
Analysis 6
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun
----------------------------------------
Analysis 7
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun
----------------------------------------
Analysis 8
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun_prop
----------------------------------------
Analysis 9
Root  : س.#.د
Lemma : سَيِّدَة
Stem  : سَيِّد
POS   : noun_prop
-------

## Root Selection Strategy

The CAMeL morphological analyzer may return multiple candidate analyses for a single query term. Rather than accepting the first analysis returned by the analyzer, the existing ranking strategy is retained and made explicitly reproducible.

The ranking algorithm assigns a score to each candidate analysis using the established criteria:

- availability of a valid root,
- agreement between the lemma and the normalized query term, and
- lexical category (part-of-speech).

The corrected implementation evaluates the complete highest-scoring set. If the highest-scoring analyses agree on one valid root, a stable lexical ordering is used only to select a representative analysis for provenance. If equally preferred analyses imply conflicting roots, the term is marked **ambiguous** and remains unresolved at this stage.

This correction does not add a new linguistic feature or disambiguation model; it prevents root selection from depending silently on the internal ordering returned by CAMeL Tools.


In [13]:
# ============================================================
# Test the Root Selection Strategy
# ============================================================

test_words = [
    "بالصبر",
    "السيدة",
    "معنى",
    "سيدنا",
    "ملكة",
    "عقوبة",
    "عقروا"
]

results = []

for word in test_words:

    normalized = hre.normalize_for_camel(word)
    analyses = analyzer.analyze(normalized)

    selection = hre.select_best_analysis_with_provenance(
        word,
        analyses
    )

    selected = selection["Selected_Analysis"]
    selected_root_validation = selection["Selected_Root_Validation"]

    results.append({

        "Query_Term": word,
        "CAMeL_Normalized_Term": normalized,
        "Analysis_Count": selection["Analysis_Count"],
        "Top_Score": selection["Top_Score"],
        "Top_Analysis_Count": selection["Top_Analysis_Count"],
        "Top_Roots": selection["Top_Roots"],
        "Top_Root_Validations": selection["Top_Root_Validations"],
        "Resolution_Status": selection["Status"],
        "Root_Resolved": selection["Root_Resolved"],
        "CAMeL_Root_Raw": (
            selected_root_validation["Raw"]
            if selected_root_validation is not None
            else None
        ),
        "CAMeL_Root_Normalized": (
            selected_root_validation["Normalized"]
            if selected_root_validation is not None
            else None
        ),
        "CAMeL_Root_Valid": (
            selected_root_validation["Valid"]
            if selected_root_validation is not None
            else False
        ),
        "CAMeL_Root_Invalid_Reason": (
            selected_root_validation["Invalid_Reason"]
            if selected_root_validation is not None
            else None
        ),
        "Selected_Root": (
            selected_root_validation["Normalized"]
            if selection["Root_Resolved"]
            and selected_root_validation is not None
            else None
        ),
        "Lemma": (
            selected.get("lex")
            if selected is not None
            else None
        ),
        "Stem": (
            selected.get("stem")
            if selected is not None
            else None
        ),
        "POS": (
            selected.get("pos")
            if selected is not None
            else None
        )

    })

results_df = pd.DataFrame(results)

print("=" * 70)
print("ROOT SELECTION STRATEGY TEST")
print("=" * 70)

display(results_df)



ROOT SELECTION STRATEGY TEST


,Query_Term,CAMeL_Normalized_Term,Analysis_Count,Top_Score,Top_Analysis_Count,Top_Roots,Top_Root_Validations,Resolution_Status,Root_Resolved,CAMeL_Root_Raw,CAMeL_Root_Normalized,CAMeL_Root_Valid,CAMeL_Root_Invalid_Reason,Selected_Root,Lemma,Stem,POS
0,بالصبر,صبر,8,170,6,[صبر],"[{'Raw': 'ص.ب.ر', 'Normalized': 'صبر', 'Valid': True, 'Invalid_Reason': None}]",RESOLVED,True,ص.ب.ر,صبر,True,NaN,صبر,صَبْر,صَبْر,noun
1,السيدة,سيدة,22,165,1,[سدة],"[{'Raw': 'سّدة', 'Normalized': 'سدة', 'Valid': True, 'Invalid_Reason': None}]",RESOLVED,True,سّدة,سدة,True,NaN,سدة,سَيِّدَة,سَيِّدَة,noun_prop
2,معنى,معني,7,70,1,[],"[{'Raw': 'ع.ن.#', 'Normalized': 'عن#', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",ANALYSIS_INVALID_ROOT,False,ع.ن.#,عن#,False,incomplete_placeholder_root,NaN,مَعْنَى,مَعْنَى,noun
3,سيدنا,سيدنا,8,25,4,[],"[{'Raw': 'س.#.د', 'Normalized': 'س#د', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",ANALYSIS_INVALID_ROOT,False,س.#.د,س#د,False,incomplete_placeholder_root,NaN,سَيِّد,سَيِّد,noun_prop
4,ملكة,ملكة,28,170,14,[ملك],"[{'Raw': 'م.ل.ك', 'Normalized': 'ملك', 'Valid': True, 'Invalid_Reason': None}]",RESOLVED,True,م.ل.ك,ملك,True,NaN,ملك,مَلَكَة,مَلَك,noun
5,عقوبة,عقوبة,7,170,7,[عقب],"[{'Raw': 'ع.ق.ب', 'Normalized': 'عقب', 'Valid': True, 'Invalid_Reason': None}]",RESOLVED,True,ع.ق.ب,عقب,True,NaN,عقب,عُقُوبَة,عُقُوب,noun
6,عقروا,عقروا,2,120,2,[عقر],"[{'Raw': 'ع.ق.ر', 'Normalized': 'عقر', 'Valid': True, 'Invalid_Reason': None}]",RESOLVED,True,ع.ق.ر,عقر,True,NaN,عقر,عَقُر,عَقُر,verb


## Applying the Hybrid Root Extraction Framework

The corrected hybrid root extraction algorithm is applied to all query terms for which the QAC stage did not provide an acceptable root.

For each fallback term, the framework performs:

1. preservation of the original term and Notebook-4 normalized term,
2. conservative normalization for CAMeL analysis,
3. morphological analysis using CAMeL Tools,
4. scoring and provenance-aware selection of the highest-ranked analysis set,
5. explicit detection of conflicting top roots, and
6. extraction of lexical evidence only when the decision is reproducible.

A CAMeL analysis without a valid root is retained as analysis evidence but is **not** counted as a successful root resolution.


In [14]:
# ============================================================
# Apply the Hybrid Root Extraction Framework
# ============================================================

camel_results = []

for _, row in fallback_df.iterrows():

    query_term = row["Normalized_Term"]
    camel_normalized_term = row["CAMeL_Normalized_Term"]

    analyses = analyzer.analyze(camel_normalized_term)

    selection = hre.select_best_analysis_with_provenance(
        query_term,
        analyses
    )

    selected = selection["Selected_Analysis"]

    selected_root_raw = None
    selected_root_normalized = None
    selected_root_valid = False
    selected_root_invalid_reason = None
    selected_lemma = None
    selected_stem = None
    selected_pos = None

    if selected is not None:
        root_validation = selection["Selected_Root_Validation"]
        selected_root_raw = root_validation["Raw"]
        selected_root_normalized = root_validation["Normalized"]
        selected_root_valid = root_validation["Valid"]
        selected_root_invalid_reason = (
            root_validation["Invalid_Reason"]
        )
        selected_lemma = selected.get("lex")
        selected_stem = selected.get("stem")
        selected_pos = selected.get("pos")

    root_resolved = bool(
        selection["Root_Resolved"]
        and selected_root_valid
        and selected_root_normalized
    )

    selected_analysis_record = None

    if selected is not None:
        selected_analysis_record = {
            "Root_Raw": selected_root_raw,
            "Root_Normalized": selected_root_normalized,
            "Root_Valid": selected_root_valid,
            "Root_Invalid_Reason": selected_root_invalid_reason,
            "Lemma": selected_lemma,
            "Stem": selected_stem,
            "POS": selected_pos
        }

    camel_results.append({

        "Dataset": row["Dataset"],
        "Question_ID": row["Question_ID"],
        "Question": row["Question"],

        "Query_Term": query_term,
        "Original_Term": row["Original_Term"],
        "Normalized_Term": row["Normalized_Term"],
        "CAMeL_Normalized_Term": camel_normalized_term,

        "QAC_Status": row["QAC_Status"],

        # Camel_Matched is retained for compatibility, but now means
        # that a usable root was actually resolved.
        "Camel_Matched": root_resolved,
        "CAMeL_Root_Resolved": root_resolved,
        "CAMeL_Analysis_Found": selection["Analysis_Found"],
        "CAMeL_Analysis_Count": selection["Analysis_Count"],
        "CAMeL_Top_Analysis_Count": selection["Top_Analysis_Count"],
        "CAMeL_Top_Roots": selection["Top_Roots"],
        "CAMeL_Top_Root_Validations": (
            selection["Top_Root_Validations"]
        ),
        "CAMeL_Root_Ambiguous": selection["Root_Ambiguous"],
        "CAMeL_Status": selection["Status"],

        "CAMeL_Root_Raw": selected_root_raw,
        "CAMeL_Root_Normalized": selected_root_normalized,
        "CAMeL_Root_Valid": selected_root_valid,
        "CAMeL_Root_Invalid_Reason": (
            selected_root_invalid_reason
        ),
        "Camel_Root": (
            selected_root_normalized
            if root_resolved
            else None
        ),
        "Lemma": selected_lemma,
        "Stem": selected_stem,
        "POS": selected_pos,

        "Analysis_Score": selection["Top_Score"],
        "CAMeL_Selected_Analysis": selected_analysis_record

    })

camel_results_df = pd.DataFrame(camel_results)

print("=" * 70)
print("HYBRID ROOT EXTRACTION COMPLETED")
print("=" * 70)

print(f"Terms Processed          : {len(camel_results_df):,}")
print(
    f"Roots Resolved by CAMeL  : "
    f"{camel_results_df['CAMeL_Root_Resolved'].sum():,}"
)
print(
    f"Roots Still Unresolved   : "
    f"{(~camel_results_df['CAMeL_Root_Resolved']).sum():,}"
)

print("\nCAMeL Resolution Status")
display(
    camel_results_df["CAMeL_Status"]
    .value_counts()
    .rename_axis("CAMeL_Status")
    .reset_index(name="Count")
)

print("\nCAMeL Invalid-Root Reasons")
display(
    camel_results_df["CAMeL_Root_Invalid_Reason"]
    .dropna()
    .value_counts()
    .rename_axis("Invalid_Reason")
    .reset_index(name="Count")
)

# Strict validity check
invalid_camel = camel_results_df[
    camel_results_df["CAMeL_Root_Resolved"]
    & (
        camel_results_df["Camel_Root"].isna()
        | ~camel_results_df["CAMeL_Root_Valid"]
    )
]

if len(invalid_camel):
    raise ValueError(
        "CAMeL validation failed: root-resolved records "
        "contain missing roots."
    )

print("\nCAMeL root-resolution validation passed.")

print("\nSample Results")
display(camel_results_df.head(10))


HYBRID ROOT EXTRACTION COMPLETED
Terms Processed          : 817
Roots Resolved by CAMeL  : 488
Roots Still Unresolved   : 329

CAMeL Resolution Status


,CAMeL_Status,Count
0,RESOLVED,488
1,ANALYSIS_INVALID_ROOT,316
2,NO_ANALYSIS,9
3,AMBIGUOUS_TOP_ROOT,4



CAMeL Invalid-Root Reasons


,Invalid_Reason,Count
0,incomplete_placeholder_root,282
1,sentinel_root,34



CAMeL root-resolution validation passed.

Sample Results


,Dataset,Question_ID,Question,Query_Term,Original_Term,Normalized_Term,CAMeL_Normalized_Term,QAC_Status,Camel_Matched,CAMeL_Root_Resolved,CAMeL_Analysis_Found,CAMeL_Analysis_Count,CAMeL_Top_Analysis_Count,CAMeL_Top_Roots,CAMeL_Top_Root_Validations,CAMeL_Root_Ambiguous,CAMeL_Status,CAMeL_Root_Raw,CAMeL_Root_Normalized,CAMeL_Root_Valid,CAMeL_Root_Invalid_Reason,Camel_Root,Lemma,Stem,POS,Analysis_Score,CAMeL_Selected_Analysis
0,Train,101,من هم قوم شعيب؟,شعيب,شعيب,شعيب,شعيب,LEXICAL_MATCH_NO_ROOT,True,True,True,7,6,[شعب],"[{'Raw': 'ش.ع.ب', 'Normalized': 'شعب', 'Valid': True, 'Invalid_Reason': None}]",False,RESOLVED,ش.ع.ب,شعب,True,NaN,شعب,شَعِيب,شَعِيب,noun,170.0,"{'Root_Raw': 'ش.ع.ب', 'Root_Normalized': 'شعب', 'Root_Valid': True, 'Root_Invalid_Reason': None, 'Lemma': 'شَعِيب', 'Stem': 'شَعِيب', 'POS': 'noun'}"
1,Train,102,من هم قوم موسى؟,موسي,موسى,موسي,موسي,LEXICAL_MATCH_NO_ROOT,False,False,True,3,1,[],"[{'Raw': 'م.#.س', 'Normalized': 'م#س', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",False,ANALYSIS_INVALID_ROOT,م.#.س,م#س,False,incomplete_placeholder_root,NaN,مُوسَى,مُوسَى,noun,70.0,"{'Root_Raw': 'م.#.س', 'Root_Normalized': 'م#س', 'Root_Valid': False, 'Root_Invalid_Reason': 'incomplete_placeholder_root', 'Lemma': 'مُوسَى', 'Ste..."
2,Train,105,من هو النبي المعروف بالصبر؟,بالصبر,بالصبر,بالصبر,صبر,NO_MATCH,True,True,True,8,6,[صبر],"[{'Raw': 'ص.ب.ر', 'Normalized': 'صبر', 'Valid': True, 'Invalid_Reason': None}]",False,RESOLVED,ص.ب.ر,صبر,True,NaN,صبر,صَبْر,صَبْر,noun,170.0,"{'Root_Raw': 'ص.ب.ر', 'Root_Normalized': 'صبر', 'Root_Valid': True, 'Root_Invalid_Reason': None, 'Lemma': 'صَبْر', 'Stem': 'صَبْر', 'POS': 'noun'}"
3,Train,106,من كفل السيدة مريم؟,السيدة,السيدة,السيدة,سيدة,NO_MATCH,True,True,True,22,1,[سدة],"[{'Raw': 'سّدة', 'Normalized': 'سدة', 'Valid': True, 'Invalid_Reason': None}]",False,RESOLVED,سّدة,سدة,True,NaN,سدة,سَيِّدَة,سَيِّدَة,noun_prop,165.0,"{'Root_Raw': 'سّدة', 'Root_Normalized': 'سدة', 'Root_Valid': True, 'Root_Invalid_Reason': None, 'Lemma': 'سَيِّدَة', 'Stem': 'سَيِّدَة', 'POS': 'n..."
4,Train,106,من كفل السيدة مريم؟,مريم,مريم,مريم,مريم,LEXICAL_MATCH_NO_ROOT,False,False,True,2,2,[],"[{'Raw': 'NTWS', 'Normalized': 'NTWS', 'Valid': False, 'Invalid_Reason': 'sentinel_root'}]",False,ANALYSIS_INVALID_ROOT,NTWS,NTWS,False,sentinel_root,NaN,مَرْيَم,مَرْيَم,noun_prop,65.0,"{'Root_Raw': 'NTWS', 'Root_Normalized': 'NTWS', 'Root_Valid': False, 'Root_Invalid_Reason': 'sentinel_root', 'Lemma': 'مَرْيَم', 'Stem': 'مَرْيَم'..."
5,Train,107,ما معنى الحطمة؟,معني,معنى,معني,معني,NO_MATCH,False,False,True,7,1,[],"[{'Raw': 'ع.ن.#', 'Normalized': 'عن#', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",False,ANALYSIS_INVALID_ROOT,ع.ن.#,عن#,False,incomplete_placeholder_root,NaN,مَعْنَى,مَعْنَى,noun,70.0,"{'Root_Raw': 'ع.ن.#', 'Root_Normalized': 'عن#', 'Root_Valid': False, 'Root_Invalid_Reason': 'incomplete_placeholder_root', 'Lemma': 'مَعْنَى', 'St..."
6,Train,108,من هو اخو سيدنا موسى؟,سيدنا,سيدنا,سيدنا,سيدنا,NO_MATCH,False,False,True,8,4,[],"[{'Raw': 'س.#.د', 'Normalized': 'س#د', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",False,ANALYSIS_INVALID_ROOT,س.#.د,س#د,False,incomplete_placeholder_root,NaN,سَيِّد,سَيِّد,noun_prop,25.0,"{'Root_Raw': 'س.#.د', 'Root_Normalized': 'س#د', 'Root_Valid': False, 'Root_Invalid_Reason': 'incomplete_placeholder_root', 'Lemma': 'سَيِّد', 'Ste..."
7,Train,108,من هو اخو سيدنا موسى؟,موسي,موسى,موسي,موسي,LEXICAL_MATCH_NO_ROOT,False,False,True,3,1,[],"[{'Raw': 'م.#.س', 'Normalized': 'م#س', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",False,ANALYSIS_INVALID_ROOT,م.#.س,م#س,False,incomplete_placeholder_root,NaN,مُوسَى,مُوسَى,noun,70.0,"{'Root_Raw': 'م.#.س', 'Root_Normalized': 'م#س', 'Root_Valid': False, 'Root_Invalid_Reason': 'incomplete_placeholder_root', 'Lemma': 'مُوسَى', 'Ste..."
8,Train,109,ما معنى القارعة؟,معني,معنى,معني,معني,NO_MATCH,False,False,True,7,1,[],"[{'Raw': 'ع.ن.#', 'Normalized': 'عن#', 'Valid': False, 'Invalid_Reaso

## Merging QAC and CAMeL Root Retrieval Results

The final stage of the hybrid root extraction framework combines the QAC evidence with the CAMeL fallback evidence while preserving one authoritative root decision for each query term.

For each query term:

- If one acceptable QAC root is available, it is retained and CAMeL is not used.
- If QAC has no acceptable root, CAMeL fallback is attempted.
- A CAMeL root is accepted only when the corrected selection procedure resolves one usable root.
- If neither route provides an acceptable root, the term remains explicitly unresolved.
- The original term, normalized term, QAC evidence, CAMeL evidence, selected root, and root source are preserved in the final record.

The resulting hybrid representation is intended to become the authoritative query-root representation consumed by subsequent semantic enrichment and RALSR stages.


In [15]:
# ============================================================
# Merge QAC and CAMeL Results
# ============================================================

merged_records = []

# Create a lookup for CAMeL fallback results
camel_lookup = {}

for _, row in camel_results_df.iterrows():

    key = (
        row["Question_ID"],
        row["Normalized_Term"]
    )

    camel_lookup[key] = row


def list_or_empty(value):
    if isinstance(value, list):
        return value
    return []


for _, row in qac_results_df.iterrows():

    question_id = row["Question_ID"]
    merged_matches = []

    for match in row["Lexical_Matches"]:

        normalized_term = match["Normalized_Term"]

        # ----------------------------------------------------
        # Authoritative QAC Root
        # ----------------------------------------------------

        if match["Root_Resolved"] and match["Root_AR"]:

            merged_matches.append({

                "Query_Term": match["Query_Term"],
                "Original_Term": match["Original_Term"],
                "Normalized_Term": normalized_term,

                "Matched": True,
                "Root_Resolved": True,
                "Resolution_Status": "RESOLVED_QAC",

                "Root_Source": "QAC",
                "Root_AR": match["Root_AR"],
                "Lemma_AR": match["Lemma_AR"],
                "POS": match["POS"],

                "QAC_Lexical_Matched": match["Lexical_Matched"],
                "QAC_Lookup_Type": match["Lookup_Type"],
                "QAC_Status": match["QAC_Status"],
                "QAC_Candidate_Count": match["QAC_Candidate_Count"],
                "QAC_Candidate_Words": list_or_empty(
                    match["QAC_Candidate_Words"]
                ),
                "QAC_Candidate_Lemmas": list_or_empty(
                    match["QAC_Candidate_Lemmas"]
                ),
                "QAC_Candidate_Roots": list_or_empty(
                    match["QAC_Candidate_Roots"]
                ),
                "QAC_Candidate_POS": list_or_empty(
                    match["QAC_Candidate_POS"]
                ),

                "CAMeL_Attempted": False,
                "CAMeL_Normalized_Term": None,
                "CAMeL_Analysis_Found": False,
                "CAMeL_Analysis_Count": 0,
                "CAMeL_Top_Analysis_Count": 0,
                "CAMeL_Top_Roots": [],
                "CAMeL_Top_Root_Validations": [],
                "CAMeL_Root_Ambiguous": False,
                "CAMeL_Status": "NOT_REQUIRED",
                "CAMeL_Root_Raw": None,
                "CAMeL_Root_Normalized": None,
                "CAMeL_Root_Valid": False,
                "CAMeL_Root_Invalid_Reason": None,
                "CAMeL_Selected_Root": None,
                "CAMeL_Selected_Lemma": None,
                "CAMeL_Selected_Stem": None,
                "CAMeL_Selected_POS": None,
                "CAMeL_Analysis_Score": None,
                "CAMeL_Selected_Analysis": None

            })

            continue

        # ----------------------------------------------------
        # CAMeL Fallback
        # ----------------------------------------------------

        key = (
            question_id,
            normalized_term
        )

        camel = camel_lookup.get(key)

        if (
            camel is not None
            and bool(camel["CAMeL_Root_Resolved"])
            and bool(camel["CAMeL_Root_Valid"])
            and pd.notna(camel["Camel_Root"])
        ):

            root_ar = camel["Camel_Root"]
            lemma_ar = camel["Lemma"]
            pos = camel["POS"]
            matched = True
            root_resolved = True
            root_source = "CAMeL"
            resolution_status = "RESOLVED_CAMEL"

        else:

            root_ar = None
            lemma_ar = None
            pos = None
            matched = False
            root_resolved = False
            root_source = "UNRESOLVED"
            resolution_status = "UNRESOLVED"

        merged_matches.append({

            "Query_Term": match["Query_Term"],
            "Original_Term": match["Original_Term"],
            "Normalized_Term": normalized_term,

            "Matched": matched,
            "Root_Resolved": root_resolved,
            "Resolution_Status": resolution_status,

            "Root_Source": root_source,
            "Root_AR": root_ar,
            "Lemma_AR": lemma_ar,
            "POS": pos,

            "QAC_Lexical_Matched": match["Lexical_Matched"],
            "QAC_Lookup_Type": match["Lookup_Type"],
            "QAC_Status": match["QAC_Status"],
            "QAC_Candidate_Count": match["QAC_Candidate_Count"],
            "QAC_Candidate_Words": list_or_empty(
                match["QAC_Candidate_Words"]
            ),
            "QAC_Candidate_Lemmas": list_or_empty(
                match["QAC_Candidate_Lemmas"]
            ),
            "QAC_Candidate_Roots": list_or_empty(
                match["QAC_Candidate_Roots"]
            ),
            "QAC_Candidate_POS": list_or_empty(
                match["QAC_Candidate_POS"]
            ),

            "CAMeL_Attempted": camel is not None,
            "CAMeL_Normalized_Term": (
                camel["CAMeL_Normalized_Term"]
                if camel is not None
                else None
            ),
            "CAMeL_Analysis_Found": (
                bool(camel["CAMeL_Analysis_Found"])
                if camel is not None
                else False
            ),
            "CAMeL_Analysis_Count": (
                int(camel["CAMeL_Analysis_Count"])
                if camel is not None
                else 0
            ),
            "CAMeL_Top_Analysis_Count": (
                int(camel["CAMeL_Top_Analysis_Count"])
                if camel is not None
                else 0
            ),
            "CAMeL_Top_Roots": (
                camel["CAMeL_Top_Roots"]
                if camel is not None
                and isinstance(camel["CAMeL_Top_Roots"], list)
                else []
            ),
            "CAMeL_Top_Root_Validations": (
                camel["CAMeL_Top_Root_Validations"]
                if camel is not None
                and isinstance(
                    camel["CAMeL_Top_Root_Validations"],
                    list
                )
                else []
            ),
            "CAMeL_Root_Ambiguous": (
                bool(camel["CAMeL_Root_Ambiguous"])
                if camel is not None
                else False
            ),
            "CAMeL_Status": (
                camel["CAMeL_Status"]
                if camel is not None
                else "NO_FALLBACK_RECORD"
            ),
            "CAMeL_Root_Raw": (
                camel["CAMeL_Root_Raw"]
                if camel is not None
                and pd.notna(camel["CAMeL_Root_Raw"])
                else None
            ),
            "CAMeL_Root_Normalized": (
                camel["CAMeL_Root_Normalized"]
                if camel is not None
                and pd.notna(camel["CAMeL_Root_Normalized"])
                else None
            ),
            "CAMeL_Root_Valid": (
                bool(camel["CAMeL_Root_Valid"])
                if camel is not None
                else False
            ),
            "CAMeL_Root_Invalid_Reason": (
                camel["CAMeL_Root_Invalid_Reason"]
                if camel is not None
                and pd.notna(
                    camel["CAMeL_Root_Invalid_Reason"]
                )
                else None
            ),
            "CAMeL_Selected_Root": (
                camel["Camel_Root"]
                if camel is not None
                and pd.notna(camel["Camel_Root"])
                else None
            ),
            "CAMeL_Selected_Lemma": (
                camel["Lemma"]
                if camel is not None
                and pd.notna(camel["Lemma"])
                else None
            ),
            "CAMeL_Selected_Stem": (
                camel["Stem"]
                if camel is not None
                and pd.notna(camel["Stem"])
                else None
            ),
            "CAMeL_Selected_POS": (
                camel["POS"]
                if camel is not None
                and pd.notna(camel["POS"])
                else None
            ),
            "CAMeL_Analysis_Score": (
                int(camel["Analysis_Score"])
                if camel is not None
                and pd.notna(camel["Analysis_Score"])
                else None
            ),
            "CAMeL_Selected_Analysis": (
                camel["CAMeL_Selected_Analysis"]
                if camel is not None
                and isinstance(
                    camel["CAMeL_Selected_Analysis"],
                    dict
                )
                else None
            )

        })

    merged_records.append({

        "Dataset": row["Dataset"],
        "Question_ID": question_id,
        "Question": row["Question"],
        "Query_Terms": row["Query_Terms"],
        "Query_Term_Provenance": row["Query_Term_Provenance"],
        "Lexical_Matches": merged_matches

    })

hybrid_results_df = pd.DataFrame(merged_records)

print("=" * 70)
print("HYBRID ROOT RETRIEVAL COMPLETED")
print("=" * 70)

print(f"Questions Processed : {len(hybrid_results_df):,}")

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

flat_matches = [
    match
    for matches in hybrid_results_df["Lexical_Matches"]
    for match in matches
]

total_terms = len(flat_matches)
resolved_terms = sum(
    bool(match["Root_Resolved"])
    for match in flat_matches
)
qac_terms = sum(
    match["Root_Source"] == "QAC"
    for match in flat_matches
)
camel_terms = sum(
    match["Root_Source"] == "CAMeL"
    for match in flat_matches
)
unresolved_terms = sum(
    match["Root_Source"] == "UNRESOLVED"
    for match in flat_matches
)

print(f"Total Query Terms : {total_terms:,}")
print(f"Resolved Roots    : {resolved_terms:,}")
print(f"QAC Roots         : {qac_terms:,}")
print(f"CAMeL Roots       : {camel_terms:,}")
print(f"Unresolved        : {unresolved_terms:,}")

if total_terms:
    print(
        f"\nRoot Coverage     : "
        f"{100 * resolved_terms / total_terms:.2f}%"
    )

# Strict lineage / status validations
invalid_resolved = [
    match
    for match in flat_matches
    if match["Root_Resolved"] and not match["Root_AR"]
]

invalid_sources = [
    match
    for match in flat_matches
    if (
        match["Root_Resolved"]
        and match["Root_Source"] not in {"QAC", "CAMeL"}
    )
]

invalid_unresolved = [
    match
    for match in flat_matches
    if (
        not match["Root_Resolved"]
        and match["Root_Source"] != "UNRESOLVED"
    )
]

if invalid_resolved or invalid_sources or invalid_unresolved:
    raise ValueError(
        "Hybrid root provenance validation failed. "
        f"resolved-without-root={len(invalid_resolved)}, "
        f"invalid-resolved-source={len(invalid_sources)}, "
        f"invalid-unresolved-source={len(invalid_unresolved)}"
    )

print("\nHybrid root provenance validation passed.")

print("\nSample Record")
display(hybrid_results_df.head(1))


HYBRID ROOT RETRIEVAL COMPLETED
Questions Processed : 251
Total Query Terms : 1,342
Resolved Roots    : 1,013
QAC Roots         : 525
CAMeL Roots       : 488
Unresolved        : 329

Root Coverage     : 75.48%

Hybrid root provenance validation passed.

Sample Record


,Dataset,Question_ID,Question,Query_Terms,Query_Term_Provenance,Lexical_Matches
0,Train,101,من هم قوم شعيب؟,"[قوم, شعيب]","[{'Token_Position': 2, 'Original_Term': 'قوم', 'Normalized_Term': 'قوم'}, {'Token_Position': 3, 'Original_Term': 'شعيب', 'Normalized_Term': 'شعيب'}]","[{'Query_Term': 'قوم', 'Original_Term': 'قوم', 'Normalized_Term': 'قوم', 'Matched': True, 'Root_Resolved': True, 'Resolution_Status': 'RESOLVED_QA..."


## Investigating the Remaining Unresolved Terms

After the QAC-first and CAMeL-fallback stages, any query term without one acceptable root remains explicitly unresolved.

This category may include:

- Modern lexical items outside the analyzer's coverage,
- named entities for which no usable root is provided,
- orthographic variants,
- misspellings or tokenization issues,
- conflicting QAC candidates,
- conflicting top CAMeL analyses, or
- other linguistic phenomena for which the current methodology does not provide a justified root decision.

No guessed root is assigned merely to increase coverage. This preserves the methodological rule that unresolved evidence must remain unresolved.


In [16]:
# ============================================================
# Analyze Remaining Unresolved Terms
# ============================================================

unresolved_records = []

for _, row in hybrid_results_df.iterrows():

    for match in row["Lexical_Matches"]:

        if not match["Root_Resolved"]:

            unresolved_records.append({

                "Dataset": row["Dataset"],
                "Question_ID": row["Question_ID"],
                "Question": row["Question"],

                "Original_Term": match["Original_Term"],
                "Normalized_Term": match["Normalized_Term"],

                "QAC_Status": match["QAC_Status"],
                "QAC_Candidate_Roots": match["QAC_Candidate_Roots"],

                "CAMeL_Status": match["CAMeL_Status"],
                "CAMeL_Top_Roots": match["CAMeL_Top_Roots"],
                "CAMeL_Top_Root_Validations": (
                    match["CAMeL_Top_Root_Validations"]
                ),
                "CAMeL_Root_Raw": match["CAMeL_Root_Raw"],
                "CAMeL_Root_Normalized": (
                    match["CAMeL_Root_Normalized"]
                ),
                "CAMeL_Root_Valid": match["CAMeL_Root_Valid"],
                "CAMeL_Root_Invalid_Reason": (
                    match["CAMeL_Root_Invalid_Reason"]
                ),
                "CAMeL_Selected_Lemma": (
                    match["CAMeL_Selected_Lemma"]
                ),
                "CAMeL_Selected_POS": (
                    match["CAMeL_Selected_POS"]
                )

            })

remaining_unresolved = pd.DataFrame(unresolved_records)

print("=" * 70)
print("REMAINING UNRESOLVED TERMS")
print("=" * 70)

print(f"Number of Unresolved Terms : {len(remaining_unresolved):,}")

if len(remaining_unresolved) == 0:

    print("\nAll query terms have an acceptable root.")

else:

    print(
        f"Unique Unresolved Terms    : "
        f"{remaining_unresolved['Normalized_Term'].nunique():,}"
    )

    print("\nFrequency of Unresolved Terms")

    frequency = (
        remaining_unresolved["Normalized_Term"]
        .value_counts()
        .reset_index()
    )

    frequency.columns = ["Normalized_Term", "Frequency"]

    display(frequency)

    print("\nDetailed Records")
    display(
        remaining_unresolved.sort_values(
            ["Normalized_Term", "Question_ID"]
        )
    )


REMAINING UNRESOLVED TERMS
Number of Unresolved Terms : 329
Unique Unresolved Terms    : 208

Frequency of Unresolved Terms


,Normalized_Term,Frequency
0,سيدنا,21
1,هناك,9
2,الانسان,8
3,موسي,7
4,معني,6
...,...,...
203,متعاونا,1
204,لدين,1
205,باحتواء,1
206,الاديان,1



Detailed Records


,Dataset,Question_ID,Question,Original_Term,Normalized_Term,QAC_Status,QAC_Candidate_Roots,CAMeL_Status,CAMeL_Top_Roots,CAMeL_Top_Root_Validations,CAMeL_Root_Raw,CAMeL_Root_Normalized,CAMeL_Root_Valid,CAMeL_Root_Invalid_Reason,CAMeL_Selected_Lemma,CAMeL_Selected_POS
139,Train,391,اتهم القرآن بأنه السبب في الدكتاتورية الإسلامية لكونه أباح سبي النساء، كيف نرد على ذلك؟,أباح,اباح,NO_MATCH,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'ب.#.ح', 'Normalized': 'ب#ح', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",ب.#.ح,ب#ح,False,incomplete_placeholder_root,أَباح,verb
165,Train,392,اتهم القرآن بأنه السبب في الدكتاتورية الإسلامية لكونه أباح ضرب النساء في حالة النشوز، كيف نرد على ذلك؟,أباح,اباح,NO_MATCH,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'ب.#.ح', 'Normalized': 'ب#ح', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",ب.#.ح,ب#ح,False,incomplete_placeholder_root,أَباح,verb
170,Train,393,اتهم القرآن بأنه السبب في الدكتاتورية الإسلامية لكونه أباح التكفير وقتال الكفار حتى يسلموا، كيف نرد على ذلك؟,أباح,اباح,NO_MATCH,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'ب.#.ح', 'Normalized': 'ب#ح', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",ب.#.ح,ب#ح,False,incomplete_placeholder_root,أَباح,verb
173,Train,396,لماذا أباح القرآن حرمة الأشهر الحرم؟,أباح,اباح,NO_MATCH,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'ب.#.ح', 'Normalized': 'ب#ح', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",ب.#.ح,ب#ح,False,incomplete_placeholder_root,أَباح,verb
65,Train,255,كم دام إبتلاء أيوب عليه السلام؟,إبتلاء,ابتلاء,NO_MATCH,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'ب.ل.#', 'Normalized': 'بل#', 'Valid': False, 'Invalid_Reason': 'incomplete_placeholder_root'}]",ب.ل.#,بل#,False,incomplete_placeholder_root,ٱِبْتِلاء,noun
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,Train,144,من هو ابو سيدنا يوسف عليه السلام؟,يوسف,يوسف,LEXICAL_MATCH_NO_ROOT,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'NTWS', 'Normalized': 'NTWS', 'Valid': False, 'Invalid_Reason': 'sentinel_root'}]",NTWS,NTWS,False,sentinel_root,يُوسِف,noun_prop
209,Development,224,لماذا ألقي سيدنا يوسف عليه السلام في الجب؟,يوسف,يوسف,LEXICAL_MATCH_NO_ROOT,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'NTWS', 'Normalized': 'NTWS', 'Valid': False, 'Invalid_Reason': 'sentinel_root'}]",NTWS,NTWS,False,sentinel_root,يُوسِف,noun_prop
52,Train,237,كيف خرج سيدنا يوسف عليه السلام من السجن؟,يوسف,يوسف,LEXICAL_MATCH_NO_ROOT,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'NTWS', 'Normalized': 'NTWS', 'Valid': False, 'Invalid_Reason': 'sentinel_root'}]",NTWS,NTWS,False,sentinel_root,يُوسِف,noun_prop
127,Train,371,هل كان سيدنا يوسف عليه السلام رسولا أم نبيا؟,يوسف,يوسف,LEXICAL_MATCH_NO_ROOT,[],ANALYSIS_INVALID_ROOT,[],"[{'Raw': 'NTWS', 'Normalized': 'NTWS', 'Valid': False, 'Invalid_Reason': 'sentinel_root'}]",NTWS,NTWS,False,sentinel_root,يُوسِف,noun_prop


## Exporting the Hybrid Root Retrieval Results

The corrected hybrid query-root representation is exported for use in the subsequent semantic enrichment stage.

The exported dataset combines:

- the selected original and normalized query terms,
- QAC lexical and candidate-root evidence,
- CAMeL fallback evidence when QAC did not provide an acceptable root,
- the final authoritative root decision,
- the root source (`QAC`, `CAMeL`, or `UNRESOLVED`), and
- explicit root-resolution status.

The historical hybrid CSV is not overwritten. The corrected output is written to the Point-2 correction-output area and must be reviewed before it is promoted for downstream use.


In [17]:
# ============================================================
# Export Hybrid Root Retrieval Results
# ============================================================

export_df = hybrid_results_df.copy()

# Convert list/dict columns to standards-compliant JSON strings.
export_df["Query_Terms"] = (
    export_df["Query_Terms"]
    .apply(
        lambda value: json.dumps(
            value,
            ensure_ascii=False,
            allow_nan=False
        )
    )
)

export_df["Query_Term_Provenance"] = (
    export_df["Query_Term_Provenance"]
    .apply(
        lambda value: json.dumps(
            value,
            ensure_ascii=False,
            allow_nan=False
        )
    )
)

export_df["Lexical_Matches"] = (
    export_df["Lexical_Matches"]
    .apply(
        lambda value: json.dumps(
            value,
            ensure_ascii=False,
            allow_nan=False
        )
    )
)

OUTPUT_FILE = (
    CORRECTED_OUTPUT_DIR /
    "QuranQA_Hybrid_Root_Retrieval_Corrected.csv"
)

export_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Export Summary
# ------------------------------------------------------------

flat_matches = [
    match
    for matches in hybrid_results_df["Lexical_Matches"]
    for match in matches
]

total_questions = len(hybrid_results_df)
total_terms = len(flat_matches)
resolved_terms = sum(
    bool(match["Root_Resolved"])
    for match in flat_matches
)
qac_terms = sum(
    match["Root_Source"] == "QAC"
    for match in flat_matches
)
camel_terms = sum(
    match["Root_Source"] == "CAMeL"
    for match in flat_matches
)
remaining_terms = sum(
    match["Root_Source"] == "UNRESOLVED"
    for match in flat_matches
)

coverage = (
    resolved_terms / total_terms * 100
    if total_terms
    else 0.0
)

print("=" * 70)
print("HYBRID ROOT RETRIEVAL RESULTS EXPORTED")
print("=" * 70)

print(f"Output File          : {OUTPUT_FILE.name}")
print(f"Location             : {OUTPUT_FILE.parent}")

print("\nDataset Summary")

print(f"Questions            : {total_questions:,}")
print(f"Total Query Terms    : {total_terms:,}")
print(f"Resolved Roots       : {resolved_terms:,}")
print(f"QAC Roots            : {qac_terms:,}")
print(f"CAMeL Roots          : {camel_terms:,}")
print(f"Remaining Unresolved : {remaining_terms:,}")
print(f"Root Coverage        : {coverage:.2f}%")

print(
    "\nPreservation rule: the historical "
    "'QuranQA_Hybrid_Root_Retrieval.csv' file is not overwritten."
)

print("\nExport completed successfully.")


HYBRID ROOT RETRIEVAL RESULTS EXPORTED
Output File          : QuranQA_Hybrid_Root_Retrieval_Corrected.csv
Location             : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance

Dataset Summary
Questions            : 251
Total Query Terms    : 1,342
Resolved Roots       : 1,013
QAC Roots            : 525
CAMeL Roots          : 488
Remaining Unresolved : 329
Root Coverage        : 75.48%

Preservation rule: the historical 'QuranQA_Hybrid_Root_Retrieval.csv' file is not overwritten.

Export completed successfully.


# Summary

This notebook implemented the second stage of the proposed hybrid Arabic root retrieval framework by processing every query term for which the QAC stage did not provide an acceptable root.

The workflow included:

- loading the corrected QAC root retrieval results generated in Notebook 5,
- extracting QAC-unresolved terms, including lexical matches without roots and ambiguous QAC candidate groups,
- preserving the original and normalized query-term provenance,
- applying lightweight rule-based normalization for CAMeL Tools,
- performing automatic morphological analysis,
- applying the existing analysis-scoring criteria with explicit tie/root-ambiguity handling,
- distinguishing analysis availability from successful root resolution,
- merging CAMeL-derived roots with unambiguous QAC-derived roots, and
- validating the final authoritative root/source status for every query term.

The normalization strategy retains the established conservative treatment of Arabic orthographic variation. The correction does not add a new model, lexicon, ranking feature, or guessed-root mechanism.

Unlike the exploratory implementation, coverage is calculated only from records that contain an actual usable root. The corrected notebook therefore computes the final QAC/CAMeL/unresolved counts dynamically after execution rather than retaining the previous hard-coded 99.58% statement.

The corrected hybrid export preserves the trace:

**original term → normalized query term → QAC evidence → CAMeL fallback evidence (when required) → selected root → root source**

and is intended to become the authoritative query-root representation for semantic enrichment in *Maqāyīs al-Lughah* and downstream RALSR processing after review and approval.
